# Phoenix Omega — Manga Mode Test Pipeline (Colab)

**What this does:** Runs the full manga render stack on a free Colab T4 GPU.  
**Time per image:** ~20-30 seconds (SD 1.5) or ~45-90 seconds (SDXL)  
**Cost:** Free  
**Spec reference:** `specs/MANGA_MODE_SYSTEM_SPEC.md` §15-17  

---

## Before you start

1. Go to **Runtime → Change runtime type → T4 GPU**
2. Run cells top to bottom
3. First run takes ~5-10 min (downloads models)
4. After that, each image is 20-30 seconds

## Step 1 — Check GPU

In [ ]:
# Verify we have a GPU
!nvidia-smi
print()

import torch
if torch.cuda.is_available():
    gpu = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_mem / 1e9
    print(f"GPU: {gpu}")
    print(f"VRAM: {vram:.1f} GB")
    print("Ready to go.")
else:
    print("NO GPU DETECTED.")
    print("Go to Runtime > Change runtime type > T4 GPU")
    print("Then re-run this cell.")

## Step 2 — Install dependencies

Takes ~2-3 minutes on first run.

In [ ]:
# Install diffusers + manga dependencies
!pip install -q diffusers transformers accelerate safetensors
!pip install -q Pillow opencv-python-headless
!pip install -q peft  # for LoRA loading
print("Dependencies installed.")

## Step 3 — Load anime/manga model

Downloads the model on first run (~2-4 GB). Cached after that.  
Using **Anything V5** (SD 1.5 anime checkpoint) — good balance of quality and speed on T4.

In [ ]:
import torch
from diffusers import StableDiffusionPipeline, DPMSolverMultistepScheduler
from PIL import Image, ImageDraw, ImageFont
import os, json, csv, time
from datetime import datetime

# -------------------------------------------------------------------
# MODEL SELECTION
# Change this to test different checkpoints from the spec (section 16.1)
# -------------------------------------------------------------------
MODEL_ID = "stablediffusionapi/anything-v5"   # SD 1.5 anime — fast on T4
# MODEL_ID = "Linaqruf/animagine-xl"          # SDXL anime — better quality, slower
# MODEL_ID = "gsdf/Counterfeit-V3.0"          # SD 1.5 — soft/cozy style

print(f"Loading model: {MODEL_ID}")
print("First run downloads ~2-4 GB. Please wait...")

pipe = StableDiffusionPipeline.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    safety_checker=None,
)
pipe.scheduler = DPMSolverMultistepScheduler.from_config(pipe.scheduler.config)
pipe = pipe.to("cuda")
pipe.enable_attention_slicing()  # saves VRAM on T4

print(f"Model loaded. Ready to render.")

## Step 4 — Define style archetypes

These are the 8 style archetypes from the spec (§5.1).  
Each one is a prompt block — no freestyle.

In [ ]:
# =====================================================================
# STYLE ARCHETYPES (from specs/MANGA_MODE_SYSTEM_SPEC.md section 5.1)
# =====================================================================

STYLE_BLOCKS = {
    "hyper_clean_cinematic": {
        "prompt": "anime, cinematic lighting, glossy textures, lens flare, detailed eyes, vibrant colors, high quality, 4k",
        "negative": "sketch, rough lines, flat colors, chibi, deformed",
    },
    "dark_psychological": {
        "prompt": "manga, black and white, high contrast, heavy ink, dark shadows, psychological, sparse background, screen tones, halftone",
        "negative": "colorful, cute, chibi, pastel, bright, happy",
    },
    "cozy_iyashikei": {
        "prompt": "anime, pastel colors, soft lighting, warm tones, rounded shapes, gentle, cozy, watercolor style, healing",
        "negative": "dark, violent, horror, sharp angles, high contrast",
    },
    "webtoon_vertical_romance": {
        "prompt": "webtoon style, full color, soft shading, fashion illustration, beautiful eyes, atmospheric background, romantic",
        "negative": "sketch, rough, dark, horror, chibi",
    },
    "meme_chaotic_humor": {
        "prompt": "manga, exaggerated expressions, rough sketch style, comedic, 4-koma style, bold lines, reaction face",
        "negative": "realistic, photographic, serious, dark",
    },
    "power_progression": {
        "prompt": "anime, dynamic action pose, energy effects, speed lines, dramatic angle, shonen style, battle aura, intense",
        "negative": "static, calm, slice of life, pastel",
    },
    "social_media_simulacra": {
        "prompt": "modern anime style, flat UI elements, smartphone screen, text message bubbles, digital aesthetic, clean lines",
        "negative": "fantasy, medieval, traditional, painterly",
    },
    "interactive_branching": {
        "prompt": "anime, clean digital art, cel shaded, choice indicators, game UI elements, visual novel style",
        "negative": "sketch, rough, painterly, dark",
    },
}

print(f"Loaded {len(STYLE_BLOCKS)} style archetypes:")
for name in STYLE_BLOCKS:
    print(f"  - {name}")

## Step 5 — Define atom-to-panel mapping

These are the camera/composition defaults from the spec (§4.1).  
Each atom type gets a deterministic panel function.

In [ ]:
# =====================================================================
# ATOM-TO-PANEL MAPPING (from spec section 4.1)
# =====================================================================

ATOM_PANEL_MAP = {
    "HOOK": {
        "panel_function": "splash_panel",
        "camera": "dramatic wide angle, low angle",
        "composition": "full page, high impact, minimal text area",
        "panels": 1,
    },
    "SCENE": {
        "panel_function": "environment_panel",
        "camera": "wide establishing shot",
        "composition": "atmospheric, mood setting, detailed background",
        "panels": 1,
    },
    "STORY": {
        "panel_function": "narrative_sequence",
        "camera": "mixed: close-up, medium, wide",
        "composition": "multi-panel sequence, escalation to turning point",
        "panels": 4,  # scales with mechanism_depth: shallow=4, moderate=5, deep=6
    },
    "REFLECTION": {
        "panel_function": "teacher_panel",
        "camera": "close-up, character focus",
        "composition": "speech bubble with insight, character-consistent",
        "panels": 1,
    },
    "EXERCISE": {
        "panel_function": "practice_spread",
        "camera": "medium shot, instructional framing",
        "composition": "step-by-step illustrated guide, somatic cues",
        "panels": 2,
    },
    "INTEGRATION": {
        "panel_function": "closing_panel",
        "camera": "pull-back or motif callback",
        "composition": "visual motif callback, forward motion, hopeful",
        "panels": 1,
    },
}

print("Atom-to-panel mapping loaded.")
total = sum(v["panels"] for v in ATOM_PANEL_MAP.values())
print(f"One full chapter cycle = {total} panels (HOOK+SCENE+STORY+REFLECTION+EXERCISE+INTEGRATION)")

## Step 6 — Visual Prompt Compiler

This is the core of manga mode.  
**Atom metadata → deterministic visual prompt.** No freestyle.  
Matches spec §6.3 prompt structure.

In [ ]:
# =====================================================================
# VISUAL PROMPT COMPILER (spec section 3.1 step 5 + section 6.3)
# This is the prototype of pearl_prime/manga/visual_prompt_compiler.py
# =====================================================================

def compile_visual_prompt(atom_type, atom_text, style_id, teacher_id,
                          mechanism_depth="moderate", cost_intensity="medium",
                          identity_stage="early", engine_type="shame"):
    """
    Compile a deterministic visual prompt from atom metadata.
    No freestyle. Every prompt block comes from config.
    """
    style = STYLE_BLOCKS[style_id]
    panel = ATOM_PANEL_MAP[atom_type]

    # CHARACTER BLOCK — from teacher identity
    # In production: loaded from config/manga/characters/{teacher_id}_packet.yaml
    character_blocks = {
        "ahjan": "young woman, dark hair, contemplative expression, simple dark clothing",
        "maat": "elegant woman, warm brown skin, flowing robes, wise gentle expression",
        "sai_maa": "serene figure, light flowing garments, peaceful aura, soft features",
        "ra": "strong figure, commanding presence, golden accents, intense focused eyes",
        "master_wu": "elderly man, traditional robes, long beard, calm knowing expression",
    }
    char_block = character_blocks.get(teacher_id, "mysterious figure, thoughtful expression")

    # CAMERA BLOCK — from atom type
    camera_block = panel["camera"]

    # SCENE BLOCK — derived from engine type
    engine_scenes = {
        "shame": "confined interior space, dim lighting, shadows on walls",
        "grief": "open empty landscape, muted tones, vast sky",
        "spiral": "disorienting angles, tilted perspective, fragmented space",
        "overwhelm": "cluttered environment, too many objects, sensory overload",
        "false_alarm": "ordinary safe space with subtle tension, normal room with one wrong detail",
        "comparison": "split scene, two contrasting spaces side by side",
        "watcher": "observing from distance, window or doorway framing, voyeuristic angle",
    }
    scene_block = engine_scenes.get(engine_type, "neutral interior, soft lighting")

    # EMOTION BLOCK — from cost_intensity + identity_stage
    intensity_map = {
        "low": "calm, subtle emotion, relaxed posture",
        "medium": "visible tension, guarded body language, uncertain expression",
        "high": "intense emotion, clenched hands, dramatic shadows, high contrast",
    }
    emotion_block = intensity_map.get(cost_intensity, "neutral expression")

    # ASSEMBLE (spec §6.3: CHARACTER + STYLE + CAMERA + SCENE + EMOTION)
    positive = f"{char_block}, {style['prompt']}, {camera_block}, {scene_block}, {emotion_block}, masterpiece, best quality"
    negative = f"extra limbs, deformed hands, bad anatomy, watermark, text, logo, {style['negative']}"

    return {
        "positive": positive,
        "negative": negative,
        "atom_type": atom_type,
        "style_id": style_id,
        "teacher_id": teacher_id,
        "panel_function": panel["panel_function"],
    }


# Test it
test = compile_visual_prompt(
    atom_type="HOOK",
    atom_text="That moment when you realize the shame was never yours to carry.",
    style_id="dark_psychological",
    teacher_id="ahjan",
    engine_type="shame",
    cost_intensity="high",
)
print("COMPILED PROMPT:")
print(f"  Positive: {test['positive'][:120]}...")
print(f"  Negative: {test['negative'][:80]}...")
print(f"  Panel: {test['panel_function']}")

## Step 7 — Render a single test panel

First image. ~20-30 seconds on T4.

In [ ]:
# =====================================================================
# RENDER SINGLE PANEL (prototype of panel_renderer.py)
# =====================================================================

def render_panel(prompt_data, seed=42, width=512, height=768, steps=25):
    """Render a single manga panel from a compiled prompt."""
    generator = torch.Generator("cuda").manual_seed(seed)

    start = time.time()
    image = pipe(
        prompt=prompt_data["positive"],
        negative_prompt=prompt_data["negative"],
        width=width,
        height=height,
        num_inference_steps=steps,
        guidance_scale=7.5,
        generator=generator,
    ).images[0]
    elapsed = time.time() - start

    print(f"Rendered in {elapsed:.1f}s | {prompt_data['panel_function']} | seed={seed}")
    return image


# Render the HOOK panel
hook_prompt = compile_visual_prompt(
    atom_type="HOOK",
    atom_text="That moment when you realize the shame was never yours to carry.",
    style_id="dark_psychological",
    teacher_id="ahjan",
    engine_type="shame",
    cost_intensity="high",
)

hook_image = render_panel(hook_prompt, seed=42)
hook_image

## Step 8 — Render a full chapter cycle (all 6 atom types)

This tests the full atom-to-panel mapping.  
10 panels total. ~4-5 minutes on T4.

In [ ]:
# =====================================================================
# FULL CHAPTER TEST — All 6 atom types
# This is what spec Phase 1 calls: "5 atoms → 10 panels"
# =====================================================================

STYLE = "dark_psychological"  # Change to test other archetypes
TEACHER = "ahjan"
ENGINE = "shame"
BASE_SEED = 1000

# Simulated atoms (in production: from compiled atom corpus)
test_atoms = [
    {"type": "HOOK",        "text": "That moment when you realize the shame was never yours.",      "cost_intensity": "high"},
    {"type": "SCENE",       "text": "A small room. One chair. Morning light through blinds.",      "cost_intensity": "low"},
    {"type": "STORY",       "text": "She stood at the door for three minutes before knocking.",    "cost_intensity": "medium"},
    {"type": "REFLECTION",  "text": "Your nervous system does not distinguish real threats.",      "cost_intensity": "medium"},
    {"type": "EXERCISE",    "text": "Place one hand on your chest. Notice the rhythm.",           "cost_intensity": "low"},
    {"type": "INTEGRATION", "text": "You walked in carrying something. You are leaving lighter.",  "cost_intensity": "low"},
]

# Create output directory
output_dir = "/content/manga_test_output"
os.makedirs(output_dir, exist_ok=True)

# Compile all prompts
panel_prompts = []
panel_id = 0
for atom in test_atoms:
    num_panels = ATOM_PANEL_MAP[atom["type"]]["panels"]
    for p in range(num_panels):
        prompt = compile_visual_prompt(
            atom_type=atom["type"],
            atom_text=atom["text"],
            style_id=STYLE,
            teacher_id=TEACHER,
            engine_type=ENGINE,
            cost_intensity=atom["cost_intensity"],
        )
        prompt["panel_id"] = f"p{panel_id:03d}"
        prompt["seed"] = BASE_SEED + panel_id  # deterministic seed (spec §6.4)
        prompt["atom_text"] = atom["text"]
        panel_prompts.append(prompt)
        panel_id += 1

print(f"Compiled {len(panel_prompts)} panels from {len(test_atoms)} atoms")
print(f"Style: {STYLE} | Teacher: {TEACHER} | Engine: {ENGINE}")
print(f"Estimated time: {len(panel_prompts) * 25}–{len(panel_prompts) * 35} seconds")
print()

# Render all panels
rendered = []
total_start = time.time()

for i, prompt in enumerate(panel_prompts):
    print(f"[{i+1}/{len(panel_prompts)}] {prompt['atom_type']} → {prompt['panel_function']}")
    img = render_panel(prompt, seed=prompt["seed"])
    path = os.path.join(output_dir, f"{prompt['panel_id']}_{prompt['atom_type'].lower()}.png")
    img.save(path)
    rendered.append({"image": img, "prompt": prompt, "path": path})

total_time = time.time() - total_start
print(f"\nDone. {len(rendered)} panels in {total_time:.0f}s ({total_time/len(rendered):.1f}s avg)")

## Step 9 — Assemble into a manga page

Prototype of `page_assembler.py`.  
Takes rendered panels → lays them out in a grid → adds text overlays.

In [ ]:
# =====================================================================
# PAGE ASSEMBLER (prototype of pearl_prime/manga/page_assembler.py)
# =====================================================================

def assemble_manga_page(panels, page_width=2480, page_height=3508, cols=2, padding=40, bg_color="white"):
    """
    Assemble panels into a manga page.
    Default: A4 at 300dpi (2480x3508px), 2 columns.
    """
    page = Image.new("RGB", (page_width, page_height), bg_color)
    draw = ImageDraw.Draw(page)

    rows = (len(panels) + cols - 1) // cols
    cell_w = (page_width - padding * (cols + 1)) // cols
    cell_h = (page_height - padding * (rows + 1)) // rows

    for i, panel_data in enumerate(panels):
        row = i // cols
        col = i % cols
        x = padding + col * (cell_w + padding)
        y = padding + row * (cell_h + padding)

        # Resize panel to fit cell
        img = panel_data["image"].copy()
        img = img.resize((cell_w, cell_h), Image.LANCZOS)

        # Paste panel
        page.paste(img, (x, y))

        # Draw panel border
        draw.rectangle([x, y, x + cell_w, y + cell_h], outline="black", width=3)

        # Add atom type label (top-left of panel)
        label = f"{panel_data['prompt']['atom_type']} | {panel_data['prompt']['panel_id']}"
        draw.rectangle([x, y, x + 280, y + 30], fill="black")
        draw.text((x + 8, y + 5), label, fill="white")

    return page


# Assemble page 1: HOOK + SCENE + first 2 STORY panels
page1 = assemble_manga_page(rendered[0:4], cols=2)
page1_path = os.path.join(output_dir, "page_01.png")
page1.save(page1_path)
print(f"Page 1 saved: {page1_path}")

# Assemble page 2: remaining STORY + REFLECTION + EXERCISE
page2 = assemble_manga_page(rendered[4:8], cols=2)
page2_path = os.path.join(output_dir, "page_02.png")
page2.save(page2_path)
print(f"Page 2 saved: {page2_path}")

# Assemble page 3: remaining EXERCISE + INTEGRATION
page3 = assemble_manga_page(rendered[8:], cols=2)
page3_path = os.path.join(output_dir, "page_03.png")
page3.save(page3_path)
print(f"Page 3 saved: {page3_path}")

# Show page 1
page1.resize((620, 877))

## Step 10 — Export as PDF

Combines all pages into a single manga PDF.

In [ ]:
# =====================================================================
# PDF EXPORT
# =====================================================================

pages = [page1, page2, page3]
pdf_path = os.path.join(output_dir, "manga_test_volume.pdf")

# Convert to RGB (PDF requires it)
rgb_pages = [p.convert("RGB") for p in pages]
rgb_pages[0].save(pdf_path, save_all=True, append_images=rgb_pages[1:])

print(f"PDF saved: {pdf_path}")
print(f"Pages: {len(pages)}")
print(f"Panels: {len(rendered)}")

# Download link (Colab)
try:
    from google.colab import files
    files.download(pdf_path)
    print("Download started.")
except:
    print(f"Not in Colab. File at: {pdf_path}")

## Step 11 — Write provenance manifest

Every render gets full audit trail (spec §3.1 step 8).

In [ ]:
# =====================================================================
# PROVENANCE (spec requirement: full audit trail)
# =====================================================================

manifest = {
    "volume_id": "manga_test_001",
    "created": datetime.now().isoformat(),
    "style_archetype": STYLE,
    "teacher_id": TEACHER,
    "engine": ENGINE,
    "model": MODEL_ID,
    "base_seed": BASE_SEED,
    "total_panels": len(rendered),
    "total_pages": len(pages),
    "panels": []
}

for r in rendered:
    manifest["panels"].append({
        "panel_id": r["prompt"]["panel_id"],
        "atom_type": r["prompt"]["atom_type"],
        "panel_function": r["prompt"]["panel_function"],
        "seed": r["prompt"]["seed"],
        "positive_prompt": r["prompt"]["positive"],
        "negative_prompt": r["prompt"]["negative"],
        "file": r["path"],
    })

manifest_path = os.path.join(output_dir, "manifest.json")
with open(manifest_path, "w") as f:
    json.dump(manifest, f, indent=2)

print(f"Manifest saved: {manifest_path}")
print(f"Panels tracked: {len(manifest['panels'])}")
print(f"Deterministic: re-run with same seeds = same images")

## Step 12 — Test a different style archetype

Change `TEST_STYLE` below and run to see the same atoms in a completely different visual style.  
This proves style lock works — same atoms, different archetype, different look.

In [ ]:
# =====================================================================
# STYLE COMPARISON — Same atom, different archetype
# =====================================================================

TEST_STYLE = "cozy_iyashikei"  # Change to any archetype from STYLE_BLOCKS

comparison_prompt = compile_visual_prompt(
    atom_type="HOOK",
    atom_text="That moment when you realize the shame was never yours to carry.",
    style_id=TEST_STYLE,
    teacher_id="ahjan",
    engine_type="shame",
    cost_intensity="high",
)

comparison_image = render_panel(comparison_prompt, seed=42)

# Show side by side
side_by_side = Image.new("RGB", (1044, 540), "white")
side_by_side.paste(hook_image.resize((512, 512)), (5, 14))
side_by_side.paste(comparison_image.resize((512, 512)), (527, 14))
draw = ImageDraw.Draw(side_by_side)
draw.text((10, 2), f"dark_psychological", fill="black")
draw.text((532, 2), f"{TEST_STYLE}", fill="black")
side_by_side

## Step 13 — Seed determinism test

Proves that same seed = same image (spec §6.4).  
This is critical for reproducibility.

In [ ]:
# =====================================================================
# DETERMINISM TEST — Same prompt + seed = identical output
# =====================================================================

prompt = compile_visual_prompt(
    atom_type="REFLECTION",
    atom_text="Your nervous system does not distinguish real threats.",
    style_id="dark_psychological",
    teacher_id="ahjan",
    engine_type="shame",
    cost_intensity="medium",
)

# Render twice with same seed
img_a = render_panel(prompt, seed=777)
img_b = render_panel(prompt, seed=777)

# Compare
import numpy as np
arr_a = np.array(img_a)
arr_b = np.array(img_b)
identical = np.array_equal(arr_a, arr_b)

print(f"Images identical: {identical}")
if identical:
    print("PASS: Deterministic rendering confirmed.")
    print("Same atoms + same seed = same visual output. Always.")
else:
    diff = np.mean(np.abs(arr_a.astype(float) - arr_b.astype(float)))
    print(f"WARN: Mean pixel difference = {diff:.4f}")
    print("(Small differences can occur due to GPU non-determinism in cuDNN)")

---

## What you just tested

| Spec section | What was tested |
|---|---|
| §4.1 Atom-to-panel mapping | All 6 atom types → panel functions |
| §5.1 Style archetypes | dark_psychological + comparison test |
| §6.3 Prompt structure | CHARACTER + STYLE + CAMERA + SCENE + EMOTION |
| §6.4 Seed control | Deterministic: same seed = same image |
| §6.5 Batch pipeline | 10 panels from CSV-like loop |
| §3.1 Provenance | Full manifest.json with audit trail |
| §8 Format | Assembled pages + PDF export |

## What's NOT tested here (needs real infra)

- LoRA loading (needs trained character LoRAs)
- SDF consistency layer (needs trained SDF models)
- ComfyUI workflow (this uses diffusers directly; production uses ComfyUI API)
- Celery/Redis job queue (not needed for <20 panels)
- EPUB generation (needs ebooklib)
- Translation overlays (needs locale pipeline)
- Gates (needs CI infrastructure)

## Next steps

1. Try different `STYLE` values in Step 8 (all 8 archetypes work)
2. Try different `TEACHER` values (ahjan, maat, sai_maa, ra, master_wu)
3. Try different `ENGINE` values (shame, grief, spiral, overwhelm, false_alarm)
4. When satisfied → move to RunPod/Vast.ai for batch production with ComfyUI